# 第7回 確率を出し、誤りの重さでしきい値を決める（解答例・教員用）


---


## 前回からの接続


前回は、心臓病（1）とそうでない（0）を線形回帰（linear regression）で当て、式の値を0.5で切って判定しました。混同行列（confusion matrix）の4つのマスを数え、見逃しと誤検出を分けて評価し、しきい値を動かすと2つが入れ替わることを確かめました。

前回の最後に、2つのことが言えずに残りました。1つは、線形回帰の式の値が0〜1をはみ出し、確率とは呼べないこと。もう1つは、しきい値をどこに決めるかです。今日は、この2つに答えます。


---


## 今日の分析目標


あなたは、前回と同じ循環器外来の医師です。検査の値から心臓病である確率を出し、どの確率から造影を勧めるかを、どう決めるかに答える。

0〜1に収まる確率を出す式（ロジスティック回帰）を作り、その確率が信じられるかを確かめます。そして、見逃しと誤検出の重さから、しきい値を決めます。

もう1つのねらいは、`fit`や`predict`の正体（クラスとメソッド）を読めるようになることです。標準化（standardization）の小さな部品を自分で書き、scikit-learnの部品と同じ形をしていることを確かめます。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、循環器外来の医師です。胸の痛みなどで受診した患者に、冠動脈造影を受けてもらうかを決めます。

前回は「心臓病」か「そうでない」かの判定だけを出しました。でも、患者に説明するときは、「あなたが心臓病である確率は、およそ3割です」のように、確率で伝えたいところです。そして、どの確率から造影を勧めるかを、理由を付けて決めたいと考えています。


### 手元のデータ

前回と同じ、米国クリーブランド・クリニックで冠動脈造影を受けた患者303人の記録です（Detranoほか, 1989。UCI機械学習リポジトリの心臓病データ）。1行が1人で、13本の検査値が並びます。答えの列は、造影の結果、心臓病（心臓の血管が50%以上狭くなっている）と分かったか（1）、そうでないか（0）です。

前処理も前回と同じです。欠測（missing data）のある6人を外し、区分の4本をダミー変数（dummy variable）にして、297人・18本の手がかりにします。


### 最後に出す答え

新しく来た患者の、心臓病である確率です。そして、その確率から造影を勧めるかどうかの判定と、そのしきい値を選んだ理由です。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- シグモイド関数で、線形（linear）の式の値を0〜1の確率に変える仕組みを説明できる
- ロジスティック回帰で心臓病である確率を出し、`predict_proba`の2つの列を読める
- `ConvergenceWarning`の警告文を読み、標準化で対処できる
- `fit`と`transform`を持つ小さなクラスを読み、`self`に覚えた値を別のメソッドで使う仕組みを説明できる
- 標準化とモデルを`make_pipeline`で1本にまとめ、交差検証（cross-validation）の各回の中で標準化できる
- 確率の幅ごとに、確率の平均と実際の割合を比べて、確率を信じてよいかを確かめられる
- ロジスティック回帰の係数を、オッズ比（何倍なりやすいか）として読める
- 見逃しと誤検出に重さを付け、損失が最も小さくなるしきい値を、表と式の両方から決められる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）

今日は、確率を出す`LogisticRegression`、標準化の部品`StandardScaler`、部品をつなぐ`make_pipeline`、交差検証の道具`cross_val_score`・`cross_val_predict`が加わります。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix, roc_auc_score


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


### お手本：前回と同じ手順で、手がかりの表を作る

- 役割：表を読み込み、列名を日本語にし、欠測の6人を外して、区分の4本をダミー変数にする。割合をそろえて訓練とテストに分ける
- 入力：データの置き場所`DATA_DIR`と、ファイル名`heart.csv`
- 出力：人数、手がかりの表の大きさ、訓練とテストの人数

前回の0節と1節のセルを、1つにまとめたものです。各列の中身は、前回のノートの表で確かめてください。


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
raw = pd.read_csv(f'{DATA_DIR}/heart.csv')
ja_names = {'age': '年齢', 'sex': '性別', 'cp': '胸痛の種類',
            'trestbps': '安静時血圧', 'chol': 'コレステロール',
            'fbs': '空腹時血糖', 'restecg': '安静時心電図',
            'thalach': '最大心拍数', 'exang': '運動時の胸痛',
            'oldpeak': 'ST低下', 'slope': 'STの傾き',
            'ca': '血管の数', 'thal': 'タリウム検査',
            'target': '心臓病'}
df = raw.rename(columns=ja_names).dropna().reset_index(drop=True)
kinds = ['胸痛の種類', '安静時心電図', 'STの傾き', 'タリウム検査']
X = pd.get_dummies(df.drop(columns='心臓病'), columns=kinds,
                   drop_first=True, dtype=int)
y = df['心臓病']
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)
y_true = y_te.values                 # 答えを、ただの並びにする
print(len(df), '人', X.shape)
print(len(X_tr), len(X_te), y_te.sum(), '人がテストの心臓病')


297人、18本の手がかりです。訓練237人、テスト60人で、テストの心臓病の人は28人です。前回と同じ分け方なので、テストの60人も前回と同じ人たちです。


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は、②の前処理と解析を1本にまとめ、③で確率を読み、しきい値を決めます。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i in (2, 3)
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


### 今日の道筋

次の6つの小さな問いに答えていきます。

1. 0〜1に収めるには？（1節）
2. 1本にまとめるには？（2節）
3. その確率は信じてよいか？（3節）
4. 係数をどう読むか？（4節）
5. どこで切るか？（5節）
6. 新しい患者には？（6節）


---


## 1. 0〜1に収めるには？


手がかりの表ができました。まず、前回の線形回帰の式の値が、どれだけ0〜1をはみ出していたかを思い出します。


### おさらい：線形回帰の式の値

- 役割：前回と同じ線形回帰の式を作り、テストの人の式の値の最小と最大、AUCを出す
- 入力：訓練の`X_tr`・`y_tr`と、テストの`X_te`
- 出力：式の値の最小と最大、AUC、テストの1人目の式の値


In [ ]:
m = LinearRegression().fit(X_tr, y_tr)
s = m.predict(X_te)                 # テストの人の、式の値
print(round(s.min(), 2), round(s.max(), 2))
print(round(roc_auc_score(y_true, s), 3))
print(round(s[0], 2))               # テストの1人目


式の値は−0.21から1.17まででした。AUCは0.929です。テストの1人目の式の値は0.33で、0.5に届かず、前回は「心臓病でない」と判定されました。直線（平面）の式は、手がかりの値しだいで、0を下回り、1を超えます。

0〜1に収めるには、式の値を、0〜1の範囲に押し込む関数に通せばよさそうです。


### シグモイド関数

どんな数$z$も、0から1の間の数に変える関数があります。シグモイド関数（sigmoid function）です。

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

$e$は自然対数の底（約2.718）です。$z$がとても大きいと$e^{-z}$は0に近づくので、$\sigma(z)$は1に近づきます。$z$がとても小さい（マイナスに大きい）と$e^{-z}$は大きくなるので、$\sigma(z)$は0に近づきます。$z = 0$のときは、ちょうど0.5です。

- 役割：$z$を−4から4まで5つ並べ、シグモイド関数に通す
- 入力：5つの数の並び`z`
- 出力：シグモイド関数の値

`np.exp(-z)`は、$e^{-z}$を、並びの1つずつに計算します。


In [ ]:
z = np.array([-4, -2, 0, 2, 4])
print((1 / (1 + np.exp(-z))).round(3))


−4は0.018に、0は0.5に、4は0.982になりました。どの値も、0と1の間に収まります。


In [ ]:
zz = np.linspace(-6, 6, 200)
plt.plot(zz, 1 / (1 + np.exp(-zz)))
plt.axhline(0.5, color='gray', ls='--')
plt.xlabel('z')
plt.ylabel('σ(z)')
plt.show()


シグモイド関数の形は、S字です。$z$が0の近くでは急に変わり、端では0や1にゆっくり近づきます。


### ロジスティック回帰

線形回帰の式の値$z_i$を、シグモイド関数に通したものを、心臓病である確率$p_i$とします。

$$
z_i = b_0 + b_1 x_{i1} + \cdots + b_{18} x_{i,18}, \qquad p_i = \sigma(z_i)
$$

この式を、ロジスティック回帰（logistic regression）と呼びます。回帰という名前ですが、確率を出して分類（classification）に使います。

係数$b_0, b_1, \ldots$は、最小二乗ではなく、尤度（likelihood）が最も大きくなるように決めます。尤度は、「その係数のもとで、実際の答えの並びが出てくる確率」です。心臓病の人（$y_i = 1$）には$p_i$を、そうでない人（$y_i = 0$）には$1 - p_i$を、全員分かけあわせます。

$$
L = \prod_{i=1}^{n} p_i^{\,y_i} (1 - p_i)^{1 - y_i}
$$

心臓病の人の$p_i$が1に近く、そうでない人の$p_i$が0に近いほど、$L$は大きくなります。この決め方を、最尤法（maximum likelihood method）と呼びます。


### お手本：ロジスティック回帰を当てる

- 役割：訓練の人で、18本の手がかりから心臓病の確率を出すロジスティック回帰の式を作る
- 入力：訓練の`X_tr`・`y_tr`
- 出力：なし（警告が出るかを見る）

使い方は、`LinearRegression`と同じで、`fit(手がかり, 答え)`です。


In [ ]:
lr = LogisticRegression().fit(X_tr, y_tr)


セルの出力に、`ConvergenceWarning`という警告が出たはずです。第5回のラッソでも見た警告です。


### 警告を読む

警告の文には、こう書いてあります。

```
lbfgs failed to converge after 100 iteration(s)
...
Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in: ...
```

「lbfgs（係数を探す計算の方法）が、100回くり返しても収束（convergence）しなかった。くり返しの回数を増やすとよい。データの尺度をそろえる（scale）ことも考えるとよい」です。

尤度を最大にする係数は、最小二乗のように1回の計算では求まりません。係数を少しずつ動かす計算をくり返して探します。列ごとに値の大きさが大きく違うと、この探し方が遠回りになります。


### 変えてみる：列の大きさを比べる

- 役割：訓練の人の、3本の列の標準偏差（standard deviation）を比べる
- 入力：訓練の`X_tr`
- 出力：コレステロール・最大心拍数・性別の標準偏差


In [ ]:
print(X_tr[['コレステロール', '最大心拍数', '性別']].std().round(2))


コレステロールの標準偏差は52.47、最大心拍数は23.03ですが、性別（0と1）は0.47です。100倍ほど違います。警告の言うとおり、尺度をそろえます。第3回と第5回の標準化です。


### お手本：標準化してから当てる

- 役割：訓練の平均と標準偏差で、訓練とテストを標準化し、ロジスティック回帰を当てて、テストの先頭3人の確率を出す
- 入力：訓練の`X_tr`・`y_tr`と、テストの`X_te`
- 出力：先頭3人の`predict_proba`の値

`standardize`は、第5回と同じ関数です。平均と標準偏差は、訓練だけで決めます（第3回）。

`predict_proba`は、区分ごとの確率を返すメソッドです。probaは確率（probability）の略です。


In [ ]:
def standardize(X, mean, std):
    return (X - mean) / std

mean_tr, std_tr = X_tr.mean(), X_tr.std()   # 訓練だけで決める
Z_tr = standardize(X_tr, mean_tr, std_tr)
Z_te = standardize(X_te, mean_tr, std_tr)
lr = LogisticRegression().fit(Z_tr, y_tr)
print(lr.predict_proba(Z_te)[:3].round(3))


今度は、警告が出ません。

`predict_proba`の結果は、1行が1人で、2つの列があります。1列目は「心臓病でない（0）」の確率、2列目は「心臓病（1）」の確率です。1行目なら0.693と0.307で、足すと1になります。心臓病の確率は、2列目を`[:, 1]`で取り出します。


### 変えてみる：心臓病の確率だけを取り出す

- 役割：テストの60人の、心臓病の確率を取り出し、最小・最大と先頭5人の値を出す
- 入力：標準化したテストの`Z_te`
- 出力：確率の最小と最大、先頭5人の確率


In [ ]:
prob = lr.predict_proba(Z_te)[:, 1]     # 2列目＝心臓病の確率
print(round(prob.min(), 3), round(prob.max(), 3))
print(prob[:5].round(2))


確率は、最小0.005、最大0.994で、すべて0〜1に収まりました。先頭の5人は0.31・0.99・0.77・0.92・0.94です。前回の線形回帰では、1人目の式の値は0.33でした。


In [ ]:
# @title 図：テストの60人の確率（答えで色分け）
rng = np.random.default_rng(0)
jit = rng.uniform(-0.25, 0.25, len(prob))
plt.scatter(prob[y_true == 0], jit[y_true == 0], label='答え：心臓病でない')
plt.scatter(prob[y_true == 1], jit[y_true == 1] + 1, label='答え：心臓病')
plt.yticks([0, 1], ['心臓病でない', '心臓病'])
plt.xlim(-0.3, 1.3)
plt.xlabel('心臓病である確率')
plt.legend(loc='upper left', fontsize=10)
plt.show()


### ロジスティック回帰にも罰がある

`LogisticRegression`には、初めから係数への罰（正則化）が付いています。第5回のリッジと同じ、係数の二乗の罰です。罰の強さは引数`C`で決まり、既定値は1.0です。

- 役割：罰の強さ`C`の値を確かめる
- 入力：ロジスティック回帰の式`lr`
- 出力：`C`の値

`C`は、第5回の`alpha`と向きが逆で、小さいほど罰が強くなります（`help(LogisticRegression)`の`C`の説明に、Inverse of regularization strengthとあります）。罰をかけるなら、第5回と同じく、先に標準化しておく必要があります。


In [ ]:
print(lr.C)


### TODO①：0.5で切って、評価する

確率`prob`を0.5で切って判定`pred`を作り、`confusion_matrix(y_true, pred)`で混同行列を表示してください。あわせて、確率`prob`のAUCを小数第3位まで表示してください。


In [ ]:
# TODO: prob を 0.5 で切って判定し、混同行列と AUC を表示する

# 解答例①：0.5で切って評価する
pred = (prob >= 0.5).astype(int)
print(confusion_matrix(y_true, pred))
print(round(roc_auc_score(y_true, prob), 3))


混同行列は`[[29, 3], [6, 22]]`で、前回の線形回帰と同じです。見逃し6人、誤検出3人です。AUCは0.944で、線形回帰の0.929より少し高くなりました。

0.5で切った判定は、線形回帰と変わりませんでした。ロジスティック回帰で変わったのは、出てくる値が、0〜1に収まる確率になったことです。


---


## 2. 1本にまとめるには？


確率が出せました。ただし、手順は2段です。訓練の平均と標準偏差で標準化してから、式に入れます。新しい患者が来るたびに、この2段を正しい順で書かなければなりません。


### 変えてみる：標準化を忘れると

- 役割：標準化していないテストの`X_te`を、そのまま式に入れて、「心臓病」と判定される人数を数える
- 入力：標準化した手がかりで作った式`lr`と、標準化していない`X_te`
- 出力：0.5以上になった人数

わざと、1段目を飛ばしてみます。


In [ ]:
p_forgot = lr.predict_proba(X_te)[:, 1]   # 標準化を忘れた
print((p_forgot >= 0.5).sum(), '人が「心臓病」')


60人のうち54人が「心臓病」と判定されました。式は、コレステロールが0のまわりにある（標準化した）値を待っているのに、200を超えるような元の値が入ったからです。エラーは出ないので、気づかないまま、おかしな確率を出してしまいます。

訓練の平均と標準偏差を覚えておき、いつも同じ手順で使う部品があれば、この誤りを防げます。


### おさらい：def

`def`は、手順に名前を付けてまとめる書き方でした（第4回）。1節の`standardize`も`def`です。

```python
def standardize(X, mean, std):
    return (X - mean) / std
```

ただ、`standardize`は、平均と標準偏差を毎回渡す必要があります。覚えておく場所がないからです。


### お手本：標準化の部品をクラスで書く

値と手順をひとまとめにする書き方が、クラス（class）です。クラスの中に書いた`def`を、メソッド（method）と呼びます。

- 役割：`fit`で訓練の平均と標準偏差を覚え、`transform`でそれを使って標準化する部品を作る
- 入力：なし（部品の設計図を書くだけ）
- 出力：なし


In [ ]:
class Standardizer:
    def fit(self, X):
        self.mean = X.mean()     # 訓練の平均を覚える
        self.std = X.std()       # 訓練の標準偏差を覚える
        return self

    def transform(self, X):
        return (X - self.mean) / self.std


ブロックで読みます。

- `class Standardizer:`：`Standardizer`という名前の部品の設計図を書く
- `def fit(self, X):`：覚えるメソッド。`self.mean`と`self.std`に、`X`の平均と標準偏差をしまう
- `def transform(self, X):`：使うメソッド。しまっておいた`self.mean`と`self.std`で標準化する

`self`は、「この部品自身」です。`self.mean = ...`と書くと、部品の中に`mean`という値がしまわれ、あとで別のメソッドから`self.mean`で取り出せます。`return self`は、部品自身を返すので、`Standardizer().fit(X_tr)`のように、作ってすぐ`fit`した部品を受け取れます。


### お手本：部品を使う

- 役割：部品を作って訓練で`fit`し、覚えた平均を見て、テストの先頭3人・3列を`transform`する
- 入力：訓練の`X_tr`と、テストの`X_te`
- 出力：覚えた年齢とコレステロールの平均、標準化したテストの先頭3人・3列

`Standardizer()`で、設計図から部品を1つ作ります。


In [ ]:
sc = Standardizer().fit(X_tr)          # 訓練で覚える
print(sc.mean[['年齢', 'コレステロール']].round(2))
print(sc.transform(X_te).iloc[:3, :3].round(3))


部品は、訓練の平均（年齢54.32歳、コレステロール248.62）を覚えています。テストを渡すと、その平均と標準偏差で標準化します。新しい患者が来ても、`sc.transform`に渡すだけです。


### 変えてみる：scikit-learnの部品と比べる

scikit-learnにも、標準化の部品`StandardScaler`があります。同じ`fit`と`transform`を持っています。

- 役割：`StandardScaler`を訓練で`fit`し、テストの先頭3人・3列を`transform`する
- 入力：訓練の`X_tr`と、テストの`X_te`
- 出力：標準化したテストの先頭3人・3列


In [ ]:
ss = StandardScaler().fit(X_tr)
print(ss.transform(X_te)[:3, :3].round(3))


ほぼ同じですが、1人目の年齢は、自分の部品で−0.921、`StandardScaler`で−0.923です。小数第3位で、少しずれています。


### お手本：ドキュメントで、標準偏差の計算を読む

`help(StandardScaler)`のNotesの節に、こう書いてあります。

```
We use a biased estimator for the standard deviation, equivalent to
`numpy.std(x, ddof=0)`.
```

「標準偏差には、偏りのある推定量を使う。`numpy.std(x, ddof=0)`と同じ」です。標準偏差は、偏差の二乗の和を、人数$n$で割るか、$n - 1$で割るかの2通りがあります。

$$
\sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2}
\quad \text{と} \quad
\sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2}
$$

`ddof=0`は$n$で割る（左）、`ddof=1`は$n - 1$で割る（右）という指定です。pandasの`std()`は、何も書かないと`ddof=1`です。`StandardScaler`は`ddof=0`なので、ずれたのです。


In [ ]:
help(StandardScaler)


### TODO②：自分の部品を、StandardScalerにそろえる

お手本の`Standardizer`のセルを写し、`X.std()`を`X.std(ddof=0)`に変えて、名前を`Standardizer0`にしてください。訓練で`fit`し、テストの先頭3人・3列を`transform`して、`StandardScaler`の結果と同じになるか確かめてください。


In [ ]:
# TODO: ddof=0 にした Standardizer0 を書き、StandardScaler と比べる

# 解答例②：ddof=0 にそろえる
class Standardizer0:
    def fit(self, X):
        self.mean = X.mean()
        self.std = X.std(ddof=0)   # n で割る
        return self

    def transform(self, X):
        return (X - self.mean) / self.std

sc0 = Standardizer0().fit(X_tr)
print(sc0.transform(X_te).iloc[:3, :3].round(3))
print(ss.transform(X_te)[:3, :3].round(3))


1人目の年齢は−0.923になり、`StandardScaler`と同じ数になります。自分で書いた部品と、scikit-learnの部品は、同じ形（`fit`で覚え、`transform`で使う）で、同じ計算をしています。ドキュメントの注にあるとおり、`ddof`の違いが予測の当たりに効くことは、ふつうはありません。


### 覚えた値は、属性で見られる

`StandardScaler`も、`fit`で覚えた値を、部品の中にしまっています。自分の部品の`self.mean`・`self.std`にあたるのが、`mean_`と`scale_`です。名前の最後の`_`は、「`fit`で覚えた値」の印です。これまで使ってきた`coef_`（係数）も同じです。

- 役割：`StandardScaler`が覚えた、先頭2列の平均と標準偏差を見る
- 入力：`fit`した`ss`
- 出力：年齢と性別の、平均と標準偏差


In [ ]:
print(ss.mean_[:2].round(2))     # 年齢、性別の平均
print(ss.scale_[:2].round(2))    # 年齢、性別の標準偏差


年齢の平均は54.32で、自分の部品が覚えた値と同じです。

モデルも同じ形をしています。`LogisticRegression`は、`fit`で係数を`coef_`に覚え、`predict_proba`でそれを使って確率を出します。scikit-learnの部品はどれも、`fit`で覚え、`transform`や`predict`で使う、という同じ形をしています。


### お手本：標準化とモデルを、1本のパイプラインに

同じ形の部品は、順につなげられます。部品を順に並べたものを、パイプライン（pipeline）と呼びます。

- 役割：`StandardScaler`と`LogisticRegression`をつないだパイプラインを作り、訓練で`fit`して、テストの先頭5人の確率を出す
- 入力：訓練の`X_tr`・`y_tr`と、テストの`X_te`（どちらも標準化していない元の値）
- 出力：テストの先頭5人の、心臓病の確率

`make_pipeline(部品1, 部品2)`で、部品を順につなぎます。


In [ ]:
pipe = make_pipeline(StandardScaler(), LogisticRegression())
pipe.fit(X_tr, y_tr)
p_te = pipe.predict_proba(X_te)[:, 1]   # 元の値を渡すだけ
print(p_te[:5].round(2))


先頭の5人の確率は0.31・0.99・0.77・0.92・0.94で、1節で手で標準化したときと、小数第2位まで同じです（標準偏差の`ddof`が違うだけです）。

パイプラインの`fit`は、次の順に部品を呼びます。

1. `StandardScaler`の`fit`で、訓練の平均と標準偏差を覚え、`transform`で訓練を標準化する
2. 標準化した訓練で、`LogisticRegression`の`fit`をする

`predict_proba`は、`StandardScaler`の`transform`で（訓練で覚えた値を使って）標準化してから、`LogisticRegression`の`predict_proba`を呼びます。標準化を忘れることは、もうありません。


In [ ]:
# @title 図：パイプラインの fit と predict_proba
fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off'); ax.set_xlim(0, 10); ax.set_ylim(0, 3)
def box(x, y0, t, c):
    ax.text(x, y0, t, ha='center', va='center', fontsize=12, color=c,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor=c, lw=1.8))
for row, (yy, a, b, name) in enumerate([(2.3, 'fit\n（平均を覚える）', 'fit\n（係数を覚える）', 'fit'),
                                        (0.7, 'transform\n（覚えた平均で）', 'predict_proba\n（覚えた係数で）', 'predict_proba')]):
    box(1.0, yy, name, '#1a1a2e')
    box(4.2, yy, 'StandardScaler\n' + a, '#0066cc')
    box(8.0, yy, 'LogisticRegression\n' + b, '#e63946')
    for x0, x1 in [(1.9, 2.8), (5.6, 6.5)]:
        ax.annotate('', xy=(x1, yy), xytext=(x0, yy), arrowprops=dict(arrowstyle='->', color='#555', lw=2))
plt.show()


### お手本：パイプラインごと、交差検証する

パイプラインは、1つのモデルのように扱えます。第5回の`cross_val_score`に、そのまま渡せます。

- 役割：パイプラインを、心臓病の割合をそろえた5つの組で交差検証し、各回のAUCと平均を出す
- 入力：パイプライン`pipe`、手がかりの表`X`と答え`y`
- 出力：5回分のAUCと、その平均

`scoring='roc_auc'`で、AUCで評価します（`scoring`の名前の一覧は、第5回で読んだドキュメントにあります）。`cv=skf`で、前回の`StratifiedKFold`の分け方を渡します。


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
auc_cv = cross_val_score(pipe, X, y, cv=skf, scoring='roc_auc')
print(auc_cv.round(3))
print(round(auc_cv.mean(), 3))


5回のAUCは0.824から0.945まで揺れ、平均は0.902です。テストの1回の0.944は、この範囲の高いほうにあたります。前回の線形回帰の交差検証のAUCと比べても、ほぼ同じ高さです。確率に変えても、心臓病の人を上に並べる力は、ほとんど変わりません。

各回の中で、`cross_val_score`は、その回の訓練でパイプラインを`fit`します。だから標準化の平均と標準偏差も、その回の訓練だけで決まります。第5回の`cv_model`で、各回の中に手で書いた標準化が、パイプラインなら1行で済みます。


---


## 3. その確率は信じてよいか？


確率は0〜1に収まり、手順も1本になりました。では、その確率は、言葉どおりに信じてよいのでしょうか。「確率0.3」と出た人たちは、本当に3割ほどが心臓病なのでしょうか。


### 確かめ方：確率の幅ごとに数える

確率がおよそ0.3と出た人を集めて、その中で実際に心臓病だった人の割合を数えます。それが0.3に近ければ、確率を信じてよさそうです。

ただ、テストは60人しかいないので、幅ごとに分けると数人ずつになってしまいます。そこで、訓練の237人で確かめます。ただし、式を作るのに使った人の確率は、当たりやすくなっています。そこで交差検証を使い、1人1人に、その人を使わずに作った式の確率を出します。


### お手本：式が見ていない人の確率を出す

- 役割：訓練の237人を5つの組に分け、各組の人の確率を、ほかの4組で作ったパイプラインで出す
- 入力：パイプライン`pipe`、訓練の`X_tr`・`y_tr`
- 出力：確率の個数と、先頭5人の確率

`cross_val_predict`は、`cross_val_score`と同じ分け方で、評価の数ではなく、1人1人の予測を返します。`method='predict_proba'`で、確率を返すように指定します。


In [ ]:
p_cv = cross_val_predict(pipe, X_tr, y_tr, cv=skf,
                         method='predict_proba')[:, 1]
print(len(p_cv))
print(p_cv[:5].round(2))


237人の全員に、その人を使わずに作った式での確率が付きました。


### お手本：確率の幅ごとに、実際の割合を比べる

- 役割：確率を0.2刻みの5つの幅に分け、幅ごとに人数と、確率の平均、実際に心臓病だった人の割合を出す
- 入力：確率`p_cv`と、訓練の答え`y_tr`
- 出力：幅ごとの人数と、確率の平均・実際の割合の表

`pd.cut(値, 区切り)`は、値を区切りで幅に分け、それぞれがどの幅に入るかを返します。`(0.2, 0.4]`は「0.2より大きく、0.4以下」です。分けたあとは、第3回からの`groupby`で幅ごとに平均します。心臓病の列（1と0）の平均は、心臓病の人の割合です。`observed=True`は、「人のいる幅だけを出す」という指定です。


In [ ]:
check = pd.DataFrame({'確率': p_cv, '心臓病': y_tr.values})
check['幅'] = pd.cut(p_cv, [0, 0.2, 0.4, 0.6, 0.8, 1.0])
print(check['幅'].value_counts(sort=False))
print(check.groupby('幅', observed=True).mean().round(3))


確率が0.8より大きいと出た70人では、確率の平均が0.95、実際に心臓病だった人の割合が0.886です。0.2以下と出た97人では、確率の平均が0.079、実際の割合が0.103です。まん中の幅は22〜26人と少ないので、確率の平均と実際の割合の差が大きめに出ています。


In [ ]:
res = check.groupby('幅', observed=True).mean()
plt.plot([0, 1], [0, 1], '--', color='gray', label='確率どおり')
plt.plot(res['確率'], res['心臓病'], 'o-', label='ロジスティック回帰')
plt.xlabel('確率の平均')
plt.ylabel('実際に心臓病だった割合')
plt.legend()
plt.show()


点は、おおむね対角線（確率どおり）に沿っています。確率の値と、実際の割合が合っていることを、確率の較正（calibration）がとれている、と言います。この確率なら、「あなたが心臓病である確率は、およそ3割です」と伝えることに、根拠があります。

前回の線形回帰の式の値は、1.17のように1を超える値があり、こうした読み方はできませんでした。


---


## 4. 係数をどう読むか？


確率は信じてよさそうです。では、どの検査値が、確率をどれだけ動かすのでしょうか。第4回では、線形回帰の係数を「その列が1増えると、価格が何ドル上がるか」と読みました。ロジスティック回帰の係数は、どう読めばよいでしょうか。


### 確率は、一定には増えない

ロジスティック回帰の確率は、$z$をS字のシグモイド関数に通したものです。$z$が1増えたとき、確率がどれだけ増えるかは、$z$が0の近くか端かで違います。そこで、確率の代わりに、オッズ（odds）で考えます。

$$
\text{オッズ} = \frac{p}{1 - p}
$$

オッズは、「なる確率」が「ならない確率」の何倍かです。確率0.5ならオッズは1、確率0.75ならオッズは3（3対1）です。

シグモイド関数の式を変形すると、オッズの対数が、ちょうど線形の式$z$になります。

$$
\log \frac{p}{1 - p} = z = b_0 + b_1 x_1 + \cdots + b_{18} x_{18}
$$

つまり、$x_j$が1増えると、オッズの対数が$b_j$増えます。オッズでいえば、$e^{b_j}$倍になります。この$e^{b_j}$を、オッズ比（odds ratio）と呼びます。


### お手本：オッズ比を並べる

- 役割：パイプラインの中のロジスティック回帰から係数を取り出し、オッズ比にして、小さい順に並べる
- 入力：`fit`したパイプライン`pipe`
- 出力：18本の列の、オッズ比

パイプラインは部品の並びなので、`pipe[-1]`で最後の部品（ロジスティック回帰）を取り出せます。`pipe[0]`なら最初の部品（`StandardScaler`）です。


In [ ]:
coef = pd.Series(pipe[-1].coef_[0], index=X.columns)
print(np.exp(coef).round(2).sort_values())


オッズ比が1より大きい列は、値が大きいほど心臓病になりやすい向き、1より小さい列は、なりにくい向きです。

いちばん大きいのは血管の数で2.88、次がタリウム検査7（回復する欠損）の2.01です。いちばん小さいのは最大心拍数の0.65です。

ただし、この式は標準化した手がかりで作っています。だから、オッズ比は「1標準偏差増えると、何倍か」です。血管の数が1標準偏差多いと、心臓病のオッズは2.88倍になります。ほかの列の値が同じなら、という条件付きです（第4回の「合わせて読む」と同じです）。


### お手本：1単位あたりに戻す

1標準偏差では、医師に伝えにくいので、元の1単位あたりに戻します。標準化した係数を、標準偏差で割れば、元の1単位あたりの係数になります（第5回と同じ戻し方です）。標準偏差は、`StandardScaler`が`scale_`に覚えています。

- 役割：血管の数の標準偏差を取り出し、血管が1本多いときのオッズ比を出す
- 入力：係数`coef`と、パイプラインの最初の部品`pipe[0]`
- 出力：血管の数の標準偏差と、1本あたりのオッズ比


In [ ]:
scale = pd.Series(pipe[0].scale_, index=X.columns)
print(round(scale['血管の数'], 3))
print(round(np.exp(coef['血管の数'] / scale['血管の数']), 2))


血管の数の標準偏差は0.959で、1本とほぼ同じです。造影の前の透視で写った太い血管が1本多いと、心臓病のオッズは3.01倍になります。


### TODO③：最大心拍数が10拍高いと

お手本をまねて、最大心拍数が10拍高いときのオッズ比を、小数第2位まで表示してください。1拍あたりの係数を10倍してから、`np.exp`に通します。


In [ ]:
# TODO: 最大心拍数が 10 拍高いときのオッズ比を表示する

# 解答例③：最大心拍数が10拍高いと
per1 = coef['最大心拍数'] / scale['最大心拍数']   # 1拍あたり
print(round(np.exp(10 * per1), 2))


オッズ比は0.83です。運動の検査で達した最大心拍数が10拍高いと、心臓病のオッズは0.83倍、つまり2割ほど小さくなります。

オッズ比は、確率が何倍になるかではありません。確率が0.3から0.6になる（2倍）とき、オッズは0.43から1.5で、3.5倍になります。患者に伝えるときは、オッズ比ではなく、6節のように、その人の確率で伝えます。


In [ ]:
print(round(0.3 / 0.7, 2), round(0.6 / 0.4, 2))
print(round((0.6 / 0.4) / (0.3 / 0.7), 1))   # オッズは何倍か


---


## 5. どこで切るか？


確率が出せて、その読み方も分かりました。残るのは、どの確率から造影を勧めるか、つまり、しきい値をどこに決めるかです。前回見たとおり、しきい値を下げれば見逃しが減り、誤検出が増えます。


### お手本：0.5で切ると

- 役割：パイプラインの確率を0.5で切って、テストの混同行列を出す
- 入力：テストの確率`p_te`と答え`y_true`
- 出力：混同行列


In [ ]:
print(confusion_matrix(y_true, (p_te >= 0.5).astype(int)))


0.5で切ると、見逃し6人、誤検出3人です。外れた人は9人ですが、医師にとって、見逃しの6人と誤検出の3人は重さが違います。


### 問い：見逃し1人は、誤検出何人分か？

見逃すと、心臓病の人が治療の機会を失います。誤検出なら、健康な人が、負担のある造影を受けます。医師として、見逃し1人の重さは、誤検出何人分だと考えますか。考えを下に書いてください。


（ここに考えを書く）


正解は、データからは決まりません。病気の重さ、検査の負担、患者の考えから、医師と患者が決めることです。ここでは、見逃し1人を、誤検出5人分の重さと置きます。

重さが決まれば、外れ方の損失（loss）を1つの数にできます。誤検出の重さを$c_{FP}$、見逃しの重さを$c_{FN}$とすると、

$$
\text{損失} = c_{FP} \times \text{FP} + c_{FN} \times \text{FN}
$$

です。今日は$c_{FP} = 1$、$c_{FN} = 5$です。損失がいちばん小さくなるしきい値を選びます。


### お手本：しきい値ごとの損失の表

しきい値も、第5回の`alpha`と同じく、データを見て「選ぶ」ものです。テストで選ぶと、テストが最後の確かめにならなくなります。そこで、3節の`p_cv`（訓練の人の、式が見ていない確率）で選びます。

- 役割：しきい値を0.1から0.9まで変え、訓練の237人の誤検出・見逃しと損失を表にする
- 入力：確率`p_cv`、訓練の答え`y_tr`、重さ`c_fp`・`c_fn`
- 出力：しきい値ごとの誤検出・見逃し・損失の表

前回の6節の、しきい値ごとの表と同じ形です。


In [ ]:
c_fp, c_fn = 1, 5                       # 見逃しは誤検出の5倍
y_cv = y_tr.values
rows = []
for t in np.arange(0.1, 1.0, 0.1):
    t = round(t, 1)
    p = (p_cv >= t).astype(int)
    fp = ((p == 1) & (y_cv == 0)).sum()
    fn = ((p == 0) & (y_cv == 1)).sum()
    rows.append({'しきい値': t, '誤検出': fp, '見逃し': fn,
                 '損失': c_fp * fp + c_fn * fn})
cost = pd.DataFrame(rows)
print(cost)


損失がいちばん小さいのは、しきい値0.1のときで89です。0.2でも91と、ほとんど変わりません。0.5では127です。見逃しを5倍重く見ると、0.5よりずっと低いしきい値がよくなります。


### 式で考える：1人あたりの損失の見込み

表の答えは、式からも出せます。心臓病である確率が$p$の患者を考えます。

- 「心臓病」と判定する（造影を勧める）と、外れるのは心臓病でなかったときで、その確率は$1 - p$。損失の見込みは$(1 - p)\,c_{FP}$
- 「心臓病でない」と判定すると、外れるのは心臓病だったときで、その確率は$p$。損失の見込みは$p\,c_{FN}$

損失の見込みが小さいほうを選びます。「心臓病」と判定したほうが小さいのは、

$$
(1 - p)\,c_{FP} < p\,c_{FN}
\quad \Longleftrightarrow \quad
p > \frac{c_{FP}}{c_{FP} + c_{FN}}
$$

のときです。この右辺が、選ぶべきしきい値$t^*$です。確率が信じられる（較正がとれている）ことが、この考え方の前提です。

- 役割：重さから、しきい値$t^*$を計算する
- 入力：重さ`c_fp`・`c_fn`
- 出力：$t^*$


In [ ]:
t_star = c_fp / (c_fp + c_fn)
print(round(t_star, 3))


$t^* = 1/6$で、0.167です。表では0.1と0.2の損失がほぼ同じでしたが、その間にある値です。


In [ ]:
# @title 図：確率 p の患者の、損失の見込み
pp = np.linspace(0, 1, 200)
plt.plot(pp, (1 - pp) * c_fp, label='「心臓病」と判定（造影を勧める）')
plt.plot(pp, pp * c_fn, label='「心臓病でない」と判定')
plt.axvline(t_star, color='gray', ls='--')
plt.text(t_star + 0.02, 3.5, f't* = {t_star:.3f}')
plt.xlabel('心臓病である確率 p')
plt.ylabel('損失の見込み')
plt.legend()
plt.show()


2本の線は、$p = 1/6$で交わります。それより右では、「心臓病」と判定する（造影を勧める）ほうが、損失の見込みが小さくなります。


### 変えてみる：0.01刻みで確かめる

- 役割：しきい値を0.01から0.99まで0.01刻みで動かし、損失を図に描いて、いちばん小さいしきい値を出す
- 入力：確率`p_cv`、訓練の答え`y_cv`、重さ
- 出力：損失の図と、損失がいちばん小さいしきい値とその損失

`np.argmin`は、並びの中でいちばん小さい値の、位置（何番目か）を返します。


In [ ]:
ts = np.arange(1, 100) / 100            # 0.01, 0.02, ..., 0.99
losses = []
for t in ts:
    p = (p_cv >= t).astype(int)
    fp = ((p == 1) & (y_cv == 0)).sum()
    fn = ((p == 0) & (y_cv == 1)).sum()
    losses.append(c_fp * fp + c_fn * fn)
plt.plot(ts, losses)
plt.axvline(t_star, color='gray', ls='--')
plt.xlabel('しきい値')
plt.ylabel('損失（訓練の237人）')
plt.show()
print(ts[np.argmin(losses)], min(losses))


損失がいちばん小さいのは、しきい値0.16のときで80です。式で出した$t^* = 0.167$と、ほぼ同じ場所です。データで選んでも、式で出しても、同じしきい値にたどり着きます。


### TODO④：重さを変えると

見逃しの重さを、誤検出の3倍（`c_fn3 = 3`）にしたときの$t^*$を計算してください。そして、お手本の表のセルを写して`c_fn`を`c_fn3`に変え、0.1〜0.9のどこで損失がいちばん小さくなるかを確かめてください。

あとのセルの`c_fn`と`t_star`を変えないように、新しい名前`c_fn3`・`t3`を使います。


In [ ]:
# TODO: c_fn3 = 3 のときの t* と、しきい値ごとの損失の表を出す

# 解答例④：見逃しが3倍重いとき
c_fn3 = 3
t3 = c_fp / (c_fp + c_fn3)
print(t3)
rows = []
for t in np.arange(0.1, 1.0, 0.1):
    t = round(t, 1)
    p = (p_cv >= t).astype(int)
    fp = ((p == 1) & (y_cv == 0)).sum()
    fn = ((p == 0) & (y_cv == 1)).sum()
    rows.append({'しきい値': t, '誤検出': fp, '見逃し': fn,
                 '損失': c_fp * fp + c_fn3 * fn})
print(pd.DataFrame(rows))


$t^*$は0.25です。表では、しきい値0.2で損失が71といちばん小さく、0.3が72で続きます。重さを3倍にすると、5倍のときより、しきい値は上がります。見逃しの重さが軽くなるほど、0.5に近づき、重さが同じ（1対1）なら$t^* = 0.5$です。


### お手本：テストで確かめる

しきい値は訓練で選びました。最後に、テストの60人で、0.5と$t^*$を比べます。

- 役割：テストの確率を0.5と$t^*$で切り、誤検出・見逃し・損失を出す
- 入力：テストの確率`p_te`と答え`y_true`、$t^*$
- 出力：しきい値ごとの誤検出・見逃し・損失


In [ ]:
for t in [0.5, t_star]:
    p = (p_te >= t).astype(int)
    fp = ((p == 1) & (y_true == 0)).sum()
    fn = ((p == 0) & (y_true == 1)).sum()
    print(round(t, 3), '誤検出', fp, '見逃し', fn,
          '損失', c_fp * fp + c_fn * fn)


しきい値0.5では、誤検出3人・見逃し6人で、損失は33です。$t^* = 0.167$では、誤検出14人・見逃し1人で、損失は19です。心臓病の28人のうち27人を造影に回せますが、そのかわり、心臓病でない人のうち14人にも造影を勧めます。見逃しを5倍重く見るなら、この入れ替えが得だ、ということです。


---


## 6. 新しい患者には？


しきい値が決まりました。最後に、新しく来た患者に、確率と判定を伝えます。ここでは、テストの1人目を、今日外来に来た患者とします。式は、この人をまだ見ていません。


### お手本：1人の確率を出す

- 役割：テストの1人目の検査値をパイプラインに渡し、心臓病である確率を出す
- 入力：1人分の表`new`と、パイプライン`pipe`
- 出力：その人の主な検査値と、心臓病である確率

`X_te.iloc[[0]]`のように、角かっこを2重にすると、1行だけの表として取り出せます。パイプラインは表を待っているので、この形で渡します。標準化は、パイプラインが訓練の平均と標準偏差で済ませます。


In [ ]:
new = X_te.iloc[[0]]                    # テストの1人目
print(new[['年齢', '最大心拍数', 'ST低下', '血管の数']])
p_new = pipe.predict_proba(new)[0, 1]
print(round(p_new, 3))


46歳で、ST低下が3.6と大きい人です。心臓病である確率は0.307です。


### お手本：確率と判定を文にする

- 役割：確率を$t^*$と比べて、造影を勧めるかを決め、確率としきい値と判定を文にする
- 入力：確率`p_new`と、しきい値`t_star`
- 出力：患者に伝える2つの文

`if`と`else`で、条件が成り立つときと成り立たないときで、入れる言葉を変えます。f-stringの`{p_new:.1%}`は、百分率で小数第1位まで、という指定です（前回のおさらい）。


In [ ]:
if p_new >= t_star:
    msg = '造影を勧めます'
else:
    msg = '造影は勧めません'
print(f'心臓病である確率は{p_new:.1%}です。')
print(f'しきい値{t_star:.1%}と比べて、{msg}。')


「心臓病である確率は30.7%です。しきい値16.7%と比べて、造影を勧めます。」という文になりました。

この人は、0.5で切っていたら「心臓病でない」と判定され、造影を勧められなかった人です。前回の線形回帰でも、式の値0.33で見逃されていました。実際の答えを見てみます。


In [ ]:
print(y_true[0])


答えは1で、この人は心臓病でした。しきい値を誤りの重さから決めたことで、見逃さずに済みました。


### TODO⑤：別の患者の確率と判定

テストの16人目（`X_te.iloc[[15]]`）を新しく来た患者として、お手本と同じように確率を出し、2つの文を表示してください。


In [ ]:
# TODO: テストの16人目の確率を出し、判定の文を表示する

# 解答例⑤：テストの16人目
new2 = X_te.iloc[[15]]
p2 = pipe.predict_proba(new2)[0, 1]
if p2 >= t_star:
    msg = '造影を勧めます'
else:
    msg = '造影は勧めません'
print(f'心臓病である確率は{p2:.1%}です。')
print(f'しきい値{t_star:.1%}と比べて、{msg}。')
print(y_true[15])


「心臓病である確率は0.5%です。しきい値16.7%と比べて、造影は勧めません。」となります。この人の答えは0で、心臓病ではありませんでした。


### 答え

18本の検査値から、標準化とロジスティック回帰を1本にしたパイプラインで、心臓病である確率を出せます。確率は0〜1に収まり、確率の幅ごとの実際の割合とも、おおむね合っています（0.8より大きい70人で、確率の平均0.95・実際0.886）。

見逃し1人を誤検出5人分と重く見るなら、造影を勧めるしきい値は$t^* = 1/6$（0.167）です。訓練の交差検証で0.01刻みに探しても、0.16で損失がいちばん小さくなりました。テストの60人では、0.5で切ると見逃し6人・誤検出3人ですが、$t^*$で切ると見逃し1人・誤検出14人になり、損失は33から19に下がります。

今日の1人目の患者は、確率30.7%で、16.7%を超えるので、造影を勧めます。


まだ言えないこと

- 重さの5倍：仮に置いた値で、データからは決まらない。病気の重さや患者の考えで変わり、変われば$t^*$も変わる
- ほかの場で使えるか：確率の較正は、この外来の患者で確かめたもの。有病率の違う場では、確率そのものがずれる
- 区分の列をそろえる手順：`get_dummies`は、渡した表にある区分だけを列にする。新しい患者1人の表に使うと、列がそろわない。今日は、テストの行（すでにそろった列）を使った


---


## 目標に答えられたか


- 今日の目標は「検査の値から心臓病である確率を出し、どの確率から造影を勧めるかを、どう決めるか」でした。答えを、確率としきい値の数で説明できますか？
- 1節で、シグモイド関数は、どんな数を、どんな範囲の数に変えましたか？
- 1節の`ConvergenceWarning`は、何を勧めていましたか？
- 2節の`Standardizer`で、`self.mean`は、どのメソッドでしまわれ、どのメソッドで使われましたか？
- 2節で、自分の部品と`StandardScaler`がずれたのはなぜでしたか？
- 2節のパイプラインは、`fit`と`predict_proba`のときに、それぞれ何をしていましたか？
- 3節で、確率を信じてよいかを、どう確かめましたか？
- 4節で、血管の数のオッズ比3.01は、何を表していましたか？
- 5節で、$t^* = 1/6$は、どこから出てきましたか？


---


## 今日の要点


- 今日、持ち帰る1文：確率は、訓練で決めた標準化とモデルを1本のパイプラインにして出す。しきい値は、見逃しと誤検出の重さから決める（見逃しが誤検出の5倍重いなら1/6）
- ロジスティック回帰は、線形の式の値をシグモイド関数に通して、0〜1の確率にする。係数は最尤法で決める
- `ConvergenceWarning`は読み飛ばさない。尺度の違う列は標準化してから当てる
- クラスは値と手順のまとまり。`fit`で`self`に覚え、`transform`や`predict`で使う。scikit-learnの部品はどれも同じ形
- `make_pipeline`で部品をつなぐと、標準化を忘れず、交差検証の各回の中でも訓練だけで標準化できる
- 確率の幅ごとに実際の割合を比べて、確率の較正を確かめる
- $e^{\text{係数}}$は、オッズ比（その列が1増えると、なりやすさのオッズが何倍か）
- 損失＝$c_{FP}$×誤検出＋$c_{FN}$×見逃し。しきい値は$t^* = c_{FP}/(c_{FP} + c_{FN})$


---


## 次回へ


今日までで、データを読み込み、前処理をして、式を作り、交差検証で評価し、結果を解釈して答えを出す、という一連の道具がそろいました。

次回の問いは、新しいデータを渡されたとき、データの理解から答えの報告までを、自分で組み立てられるか、です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

前回の課題と同じ、年齢・最大心拍数・ST低下の3本の列だけで、標準化とロジスティック回帰のパイプラインを作ってください。訓練で`fit`し、テストの確率を、本編の$t^*$（`t_star`）で切って、TP・FP・FNと再現率、そして確率のAUCを表示してください。


In [ ]:
cols3 = ['年齢', '最大心拍数', 'ST低下']
pipe3 = make_pipeline(StandardScaler(), LogisticRegression())
pipe3.fit(X_tr[cols3], y_tr)
p3 = pipe3.predict_proba(X_te[cols3])[:, 1]
for t in [0.5, t_star]:
    q = (p3 >= t).astype(int)
    tp3 = ((q == 1) & (y_true == 1)).sum()
    fp3 = ((q == 1) & (y_true == 0)).sum()
    fn3 = ((q == 0) & (y_true == 1)).sum()
    print(round(t, 3), tp3, fp3, fn3,
          round(tp3 / (tp3 + fn3), 3), fp3 + 5 * fn3)
print(round(roc_auc_score(y_true, p3), 3))   # AUC


<details><summary>詰まったら</summary>

3本の列の名前を`cols3 = ['年齢', '最大心拍数', 'ST低下']`にしまい、`make_pipeline(StandardScaler(), LogisticRegression())`を`X_tr[cols3]`で`fit`します。
テストの確率は`predict_proba(X_te[cols3])[:, 1]`です。判定と数え方は、前回の3節のセルと同じです。

</details>


### 応用②（判断）

応用①の3本の列のパイプラインで、$t^*$で切ったときの再現率はいくつですか。小数第3位まで答えてください（例: 0.123）。


In [ ]:
q = (p3 >= t_star).astype(int)
tp3 = ((q == 1) & (y_true == 1)).sum()
fn3 = ((q == 0) & (y_true == 1)).sum()
print('答え:', round(tp3 / (tp3 + fn3), 3))


<details><summary>詰まったら</summary>

前回の課題では、3本の線形回帰の式を0.5で切ると、再現率は0.536でした。

</details>


### 応用③（解釈）

応用①の3本の列のパイプラインで、0.5で切る場合と$t^*$で切る場合を比べてください。見逃しと誤検出の人数、損失（誤検出＋5×見逃し）を使って、どちらで切ることを勧めるかを、同僚の医師に向けて3行程度で書いてください。0.5で切ったときの数も、自分で出して確かめましょう。


（模範例）

年齢・最大心拍数・ST低下の3本のパイプラインは、0.5で切ると、心臓病の28人のうち12人を見逃し（再現率0.571）、誤検出は7人で、損失は67です。

$t^*$（0.167）で切ると、見逃しは2人（再現率0.929）に減り、誤検出は23人に増えますが、損失は33に下がります。

見逃しを誤検出の5倍重く見るなら、3本の式でも$t^*$で切ることを勧めます。ただし、AUCは0.691で18本の式（0.944）より低く、同じ見逃しの少なさのために、より多くの健康な人に造影を勧めることになります。


---


## 発展（任意）


### 罰の強さCを、交差検証で選ぶ

1節で見たとおり、`LogisticRegression`には罰の強さ`C`があり、既定値は1.0でした。第5回では`alpha`を自分の`for`で選びましたが、scikit-learnには、候補を全部試して交差検証で選ぶ道具`GridSearchCV`があります。

初めての道具なので、`help(GridSearchCV)`で、引数`param_grid`の説明を読んでから使います。パイプラインの中の部品の引数は、「部品の名前`__`引数の名前」（下線2本）で指定します。部品の名前は、`make_pipeline`が小文字のクラス名で付けます（`logisticregression`）。

- 役割：`C`を0.01〜100の5つから、訓練の交差検証のAUCで選ぶ
- 入力：パイプライン`pipe`、訓練の`X_tr`・`y_tr`、分け方`skf`
- 出力：選ばれた`C`、候補ごとのAUCの平均、選んだ式のテストのAUC


In [ ]:
from sklearn.model_selection import GridSearchCV
grid = {'logisticregression__C': [0.01, 0.1, 1, 10, 100]}
gs = GridSearchCV(pipe, grid, cv=skf, scoring='roc_auc')
gs.fit(X_tr, y_tr)
print(gs.best_params_)
print(gs.cv_results_['mean_test_score'].round(3))
print(round(roc_auc_score(y_true, gs.predict_proba(X_te)[:, 1]), 3))


選ばれたのは`C`=0.01（いちばん強い罰）で、交差検証のAUCは0.91です。ただし、`C`=1の0.899との差は小さく、前回見た評価の揺れの中に入ります。テストのAUCは0.926で、`C`=1の0.944より低くなりました。

`GridSearchCV`は、最後に、選んだ`C`で訓練全体を使って作り直します。だから`gs.predict_proba`がそのまま使えます。差が揺れの中にあるときは、どの`C`を選んでも大きくは変わらない、と読むのが正直です。


### 線形判別分析でも、確率を出す

前回の発展で、線形判別分析（LDA）の係数の向きが、0/1を当てた線形回帰と同じになることを見ました。線形判別分析は、心臓病の人とそうでない人の、それぞれの手がかりの分布（正規分布）を仮定して、確率も出せます。

- 役割：線形判別分析で、テストの人の心臓病の確率を出し、範囲・AUC・ロジスティック回帰の確率との相関を見る
- 入力：訓練の`X_tr`・`y_tr`、テストの`X_te`、ロジスティック回帰の確率`p_te`
- 出力：確率の最小と最大、AUC、2つの確率の相関係数、先頭5人の確率


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
lda = LinearDiscriminantAnalysis().fit(X_tr, y_tr)
p_lda = lda.predict_proba(X_te)[:, 1]
print(round(p_lda.min(), 3), round(p_lda.max(), 3))
print(round(roc_auc_score(y_true, p_lda), 3))
print(round(np.corrcoef(p_lda, p_te)[0, 1], 3))
print(p_lda[:5].round(2))


線形判別分析の確率も0.002〜0.997で、0〜1に収まります。AUCは0.929で、ロジスティック回帰の確率との相関係数は0.992と、ほとんど同じ並び方です。

ただ、1人目の患者の確率は、ロジスティック回帰の0.31に対して、線形判別分析では0.19です。どちらも$t^*$の0.167は超えますが、確率の値そのものは、仮定の違う式で変わります。どの式の確率を患者に伝えるかは、3節のような較正の確かめで選びます。

試すなら、3節と同じ幅ごとの表を、線形判別分析の確率でも作ってみましょう。対角線に沿うのは、どちらの確率でしょうか。
